[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Imbalanced Classes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_score, recall_score,
                             roc_auc_score)

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' labels, models and helpers, rebuilt here from the same made-up stations so that each
answer runs on its own.

In [2]:
hard_train = (train["tmin"] < -15).astype(int)
hard_valid = (valid["tmin"] < -15).astype(int)
hard_test = (test["tmin"] < -15).astype(int)
ratio = (hard_train == 0).sum() / hard_train.sum()
THRESHOLDS = np.round(np.arange(0.01, 0.51, 0.01), 2)


def report(name, labels, probability, threshold=0.5):
    flagged = (probability > threshold).astype(int)
    print(f"{name:20} flags {flagged.sum():3} | precision {precision_score(labels, flagged, zero_division=0):.3f} "
          f"recall {recall_score(labels, flagged):.3f} F1 {f1_score(labels, flagged):.3f} | "
          f"average precision {average_precision_score(labels, probability):.3f} AUC {roc_auc_score(labels, probability):.3f}")


def best_threshold(labels, probability):
    scores = [f1_score(labels, (probability > threshold).astype(int)) for threshold in THRESHOLDS]
    return THRESHOLDS[int(np.argmax(scores))], max(scores)


lightgbm_plain = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1)
lightgbm_plain = lightgbm_plain.fit(train[FEATURES], hard_train)

**1.** Print the precision, recall and F1 of the plain LightGBM model at a threshold of 0.1 on the validation stations, with `report`.

In [3]:
report("LightGBM at 0.1", hard_valid, lightgbm_plain.predict_proba(valid[FEATURES])[:, 1], threshold=0.1)

LightGBM at 0.1      flags  24 | precision 0.583 recall 0.368 F1 0.452 | average precision 0.599 AUC 0.988


At 0.1 the model flags 24 days, with a precision of 0.583, a recall of 0.368 and an F1 of 0.452, the line of the worked examples' table at that threshold.

**2.** Train an `LGBMClassifier` with `is_unbalance=True` in place of `scale_pos_weight`, and print its line of `report` on the validation stations.

In [4]:
unbalanced = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1, is_unbalance=True)
unbalanced.fit(train[FEATURES], hard_train)
report("LightGBM is_unbalance", hard_valid, unbalanced.predict_proba(valid[FEATURES])[:, 1])

LightGBM is_unbalance flags  25 | precision 0.600 recall 0.395 F1 0.476 | average precision 0.608 AUC 0.987


`is_unbalance=True` gives the same line as `scale_pos_weight` set to 61.5, 25 days flagged and an average precision of 0.608, since LightGBM weights the rare class by that ratio itself.

**3.** Choose a threshold for `xgboost_weighted` with `best_threshold` on the validation stations, and print its F1 on the test stations once.

In [5]:
xgboost_weighted = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=0, scale_pos_weight=ratio)
xgboost_weighted.fit(train[FEATURES], hard_train)
threshold, valid_f1 = best_threshold(hard_valid, xgboost_weighted.predict_proba(valid[FEATURES])[:, 1])
test_flags = (xgboost_weighted.predict_proba(test[FEATURES])[:, 1] > threshold).astype(int)
print(f"threshold {threshold} | validation F1 {valid_f1:.3f} | test F1 {f1_score(hard_test, test_flags):.3f}")

threshold 0.36 | validation F1 0.576 | test F1 0.622


The weighted XGBoost model's best threshold on the validation stations is 0.36, higher than the plain model's 0.09, and its test F1 is 0.622. The weights raised the probabilities the model gives, so the threshold that balances precision and recall is higher too.

**4.** Make labels for a lowest temperature below −12 under new names, leaving `hard_train` and `hard_valid` as they are, and print the share of training days they cover and the average precision on the validation stations of a plain LightGBM model trained on them.

In [6]:
below_12_train = (train["tmin"] < -12).astype(int)
below_12_valid = (valid["tmin"] < -12).astype(int)
model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1)
model.fit(train[FEATURES], below_12_train)
print("share of training days:", round(below_12_train.mean(), 4))
print("average precision:", round(average_precision_score(below_12_valid, model.predict_proba(valid[FEATURES])[:, 1]), 3))

share of training days: 0.0539
average precision: 0.813


Below −12 covers 5.39% of the training days, more than three times the share below −15, and the plain model's average precision on the validation stations is 0.813, against 0.599 for hard frost below −15.

**5.** For `xgboost_plain`, print the mean probability it gives the validation days that had hard frost and the days that did not.

In [7]:
xgboost_plain = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=0)
xgboost_plain.fit(train[FEATURES], hard_train)
probability = xgboost_plain.predict_proba(valid[FEATURES])[:, 1]
print("hard-frost days:", round(probability[hard_valid == 1].mean(), 3))
print("other days:", round(probability[hard_valid == 0].mean(), 3))

hard-frost days: 0.361
other days: 0.003


XGBoost gives the validation days with hard frost a mean probability of 0.361 and the other days 0.003, so it sets the two kinds of day well apart although few hard-frost days pass 0.5.

---

&#8592; **Back to:** [Imbalanced Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/11-imbalanced-classes.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
